In [25]:
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from torch import tensor

### Simple Model

In [5]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 3)
        self.fc2 = nn.Linear(3, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

model = MyModel()
state_dict = {
    "fc1.weight": torch.tensor([
        [0.1, 0.2, 0.3, 0.1],
        [0.2, 0.3, 0.1, 0.2],
        [0.3, 0.1, 0.2, 0.3],
    ]),

    "fc1.bias": torch.tensor([
        0.1, 0.2, 0.3
    ]),

    "fc2.weight": torch.tensor([
        [0.1, 0.2, 0.3],
        [0.2, 0.3, 0.1],
    ]),

    "fc2.bias": torch.tensor([
        0.1, 0.2
    ]),
}

model.load_state_dict(state_dict)

<All keys matched successfully>

In [6]:
config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["fc1", "fc2"],
)

In [7]:
lora_model = get_peft_model(model, config)

In [9]:
lora_model.print_trainable_parameters()

trainable params: 12 || all params: 35 || trainable%: 34.2857


In [11]:
all_param = 0
for p in model.parameters():
    all_param += p.numel()
all_param

35

In [15]:
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        print(name, p)

base_model.model.fc1.lora_A.default.weight Parameter containing:
tensor([[ 0.0287, -0.3628, -0.2336, -0.3378]], requires_grad=True)
base_model.model.fc1.lora_B.default.weight Parameter containing:
tensor([[0.],
        [0.],
        [0.]], requires_grad=True)
base_model.model.fc2.lora_A.default.weight Parameter containing:
tensor([[ 0.2117, -0.0078, -0.0065]], requires_grad=True)
base_model.model.fc2.lora_B.default.weight Parameter containing:
tensor([[0.],
        [0.]], requires_grad=True)


In [19]:
lora_model.base_model.model.fc1.lora_B.default.weight @ lora_model.base_model.model.fc1.lora_A.default.weight 

tensor([[0., -0., -0., -0.],
        [0., -0., -0., -0.],
        [0., -0., -0., -0.]], grad_fn=<MmBackward0>)

### Forward

In [48]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(10, 4, bias=False)
        
    def forward(self, x):
        return self.fc1(x)

In [50]:
state_dict = {
    "fc1.weight": torch.tensor([
        [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        [1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0],
        [2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0],
        [3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 4.0],
    ])
}

In [51]:
model = MyModel()
model.load_state_dict(state_dict)

<All keys matched successfully>

In [52]:
lora_model = MyModel()
lora_model.load_state_dict(state_dict)
config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["fc1"],
)
lora_model = get_peft_model(lora_model, config)

### Simple Inference

In [53]:
x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
model(x)

tensor([ 3.8500,  9.3500, 14.8500, 20.3500], grad_fn=<SqueezeBackward4>)

In [54]:
x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
lora_model(x)

tensor([ 3.8500,  9.3500, 14.8500, 20.3500], grad_fn=<AddBackward0>)

### Train

In [88]:
model = MyModel()
model.load_state_dict(state_dict)

loss_fn = nn.MSELoss()

x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
model.train()
y_pred = model(x)
y_target = tensor([8.0,  20.0, 25.0, 23.0])
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

loss = loss_fn(y_pred, y_target)
loss.backward()
optimizer.step()

In [89]:
model.fc1.weight.grad

tensor([[-0.2075, -0.4150, -0.6225, -0.8300, -1.0375, -1.2450, -1.4525, -1.6600,
         -1.8675, -2.0750],
        [-0.5325, -1.0650, -1.5975, -2.1300, -2.6625, -3.1950, -3.7275, -4.2600,
         -4.7925, -5.3250],
        [-0.5075, -1.0150, -1.5225, -2.0300, -2.5375, -3.0450, -3.5525, -4.0600,
         -4.5675, -5.0750],
        [-0.1325, -0.2650, -0.3975, -0.5300, -0.6625, -0.7950, -0.9275, -1.0600,
         -1.1925, -1.3250]])

In [90]:
lora_model = MyModel()
lora_model.load_state_dict(state_dict)
config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["fc1"],
)
lora_model = get_peft_model(lora_model, config)

loss_fn = nn.MSELoss()

x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
lora_model.train()
y_pred = lora_model(x)
y_target = tensor([8.0,  20.0, 25.0, 23.0])
optimizer = torch.optim.SGD(
    lora_model.parameters(),
    lr=0.9
)

loss = loss_fn(y_pred, y_target)
loss.backward()
optimizer.step()

In [91]:
lora_model.fc1.weight.grad

In [98]:
lora_model.fc1.weight

Parameter containing:
tensor([[0.1000, 0.2000, 0.3000, 0.4000, 0.5000, 0.6000, 0.7000, 0.8000, 0.9000,
         1.0000],
        [1.1000, 1.2000, 1.3000, 1.4000, 1.5000, 1.6000, 1.7000, 1.8000, 1.9000,
         2.0000],
        [2.1000, 2.2000, 2.3000, 2.4000, 2.5000, 2.6000, 2.7000, 2.8000, 2.9000,
         3.0000],
        [3.1000, 3.2000, 3.3000, 3.4000, 3.5000, 3.6000, 3.7000, 3.8000, 3.9000,
         4.0000]])

In [92]:
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        print(name, p)

base_model.model.fc1.lora_A.default.weight Parameter containing:
tensor([[-0.2299, -0.0178, -0.1945, -0.1856, -0.2570,  0.0921, -0.0901,  0.0615,
         -0.0318,  0.0443]], requires_grad=True)
base_model.model.fc1.lora_B.default.weight Parameter containing:
tensor([[-0.4306],
        [-1.1050],
        [-1.0531],
        [-0.2750]], requires_grad=True)


In [93]:
lora_model.base_model.model.fc1.lora_A.default.weight.grad

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [94]:
lora_model.base_model.model.fc1.lora_B.default.weight.grad

tensor([[0.4784],
        [1.2278],
        [1.1702],
        [0.3055]])

In [95]:
x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
lora_model(x)

tensor([ 3.9493,  9.6048, 15.0928, 20.4134], grad_fn=<AddBackward0>)

In [106]:
lora_mat = lora_model.base_model.model.fc1.lora_B.default.weight @ lora_model.base_model.model.fc1.lora_A.default.weight
lora_mat

tensor([[ 0.0990,  0.0077,  0.0837,  0.0799,  0.1107, -0.0397,  0.0388, -0.0265,
          0.0137, -0.0191],
        [ 0.2540,  0.0197,  0.2149,  0.2051,  0.2840, -0.1018,  0.0996, -0.0680,
          0.0352, -0.0490],
        [ 0.2421,  0.0187,  0.2048,  0.1955,  0.2707, -0.0970,  0.0949, -0.0648,
          0.0335, -0.0467],
        [ 0.0632,  0.0049,  0.0535,  0.0510,  0.0707, -0.0253,  0.0248, -0.0169,
          0.0088, -0.0122]], grad_fn=<MmBackward0>)

In [107]:
x @ (lora_model.fc1.weight.T + lora_mat.T)  

tensor([ 3.9493,  9.6048, 15.0928, 20.4134], grad_fn=<SqueezeBackward4>)

In [108]:
lora_mat.shape

torch.Size([4, 10])

In [109]:
lora_model.fc1.weight.shape

torch.Size([4, 10])

### Merge Model

In [110]:
merged_model = lora_model.merge_and_unload()

In [111]:
merged_model

MyModel(
  (fc1): Linear(in_features=10, out_features=4, bias=False)
)

In [112]:
for name, p in merged_model.named_parameters():
    print(name, p)

fc1.weight Parameter containing:
tensor([[0.1990, 0.2077, 0.3837, 0.4799, 0.6107, 0.5603, 0.7388, 0.7735, 0.9137,
         0.9809],
        [1.3540, 1.2197, 1.5149, 1.6051, 1.7840, 1.4982, 1.7996, 1.7320, 1.9352,
         1.9510],
        [2.3421, 2.2187, 2.5048, 2.5955, 2.7707, 2.5030, 2.7949, 2.7352, 2.9335,
         2.9533],
        [3.1632, 3.2049, 3.3535, 3.4510, 3.5707, 3.5747, 3.7248, 3.7831, 3.9088,
         3.9878]])


In [113]:
x = tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
merged_model(x)

tensor([ 3.9493,  9.6048, 15.0928, 20.4134])